# 🤖 Model Training, Evaluation & Risk Scoring Pipeline
**Adaptive MLOps for Vulnerability Risk Prioritization**  
*Model Architecture: Calibrated Gradient Boosting & Random Forest + Feature Importance + Risk Engine*

## 1. Environment Setup & Core Imports

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    sys.path.insert(0, str(project_root / "src"))

# ML framework imports
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score

try:
    import xgboost as xgb
except ImportError:
    xgb = None

try:
    import shap
except ImportError:
    shap = None

# Custom modules
from src.models.severity_classifier import SeverityClassifier
from src.models.risk_scorer import RiskScorer
from src.models.explainer import SHAPExplainer

print("✅ ML Frameworks and Custom Models Successfully Imported!")

✅ ML Frameworks and Custom Models Successfully Imported!


## 2. Load Processed Dataset & Inspect Columns

In [2]:
data_path = project_root / "data" / "processed" / "features.parquet"
if not data_path.exists():
    print("⚠️ Parquet file not found. Generating preprocessed features dataset...")
    from scripts.seed_data import generate_realistic_seed_dataset
    from src.features.feature_engineer import FeatureEngineer
    raw_path = generate_realistic_seed_dataset(num_samples=1500, output_dir=str(project_root / "data" / "raw"))
    raw_df = pd.read_parquet(raw_path)
    features_df = FeatureEngineer().fit_transform(raw_df)
    features_df.to_parquet(data_path, index=False)
else:
    features_df = pd.read_parquet(data_path)

print(f"✓ Dataset Loaded! Total rows: {len(features_df)}")
print("\n=== Checking Column Names ===")
print(features_df.columns.tolist())

print("\n=== Printing First 10 Rows ===")
print(features_df.head(10))

✓ Dataset Loaded! Total rows: 2500

=== Checking Column Names ===
['cve_id', 'feature_version', 'av_network', 'av_adjacent', 'av_local', 'av_physical', 'ac_low', 'ac_high', 'pr_none', 'pr_low', 'pr_high', 'ui_none', 'ui_required', 'scope_changed', 'ci_none', 'ci_low', 'ci_high', 'ii_none', 'ii_low', 'ii_high', 'ai_none', 'ai_low', 'ai_high', 'has_cvss_v3', 'days_since_published', 'quarter_published', 'year_published', 'is_recent', 'in_cisa_kev', 'epss_score', 'epss_percentile', 'has_exploit_reference', 'description_length', 'description_word_count', 'has_rce_keyword', 'has_sqli_keyword', 'has_xss_keyword', 'has_overflow_keyword', 'has_auth_bypass_keyword', 'has_priv_esc_keyword', 'has_dos_keyword', 'is_memory_safety_cwe', 'is_injection_cwe', 'is_auth_cwe', 'is_config_cwe', 'cwe_category_id', 'vendor_count', 'reference_count', 'has_patch_reference', 'severity_label', 'severity_numeric']

=== Printing First 10 Rows ===
           cve_id feature_version  ...  severity_label  severity_nume

## 3. Target Distribution & Temporal Train-Test Split

In [3]:
# Determine target column
target_col = "severity_numeric" if "severity_numeric" in features_df.columns else "cvss_severity_class"
print(f"Target Column: '{target_col}'")
print("Class Distribution:")
print(features_df[target_col].value_counts(normalize=True))

# Temporal split (train on earlier published CVEs, test on recent CVEs)
if "published_date" in features_df.columns:
    features_df = features_df.sort_values("published_date")

drop_cols = ["cve_id", "published_date", "last_modified", "cvss_vector", "description", "vendor", "product", target_col, "severity_label", "epss_score", "in_cisa_kev"]
feature_cols = [c for c in features_df.columns if c not in drop_cols]

X = features_df[feature_cols].fillna(0)
y = features_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"\nTrain Set: {X_train.shape[0]} samples | Test Set: {X_test.shape[0]} samples")
print(f"Feature Matrix Dimension: {X_train.shape[1]} features")

Target Column: 'severity_numeric'
Class Distribution:
severity_numeric
2    0.5196
3    0.3656
4    0.0896
1    0.0252
Name: proportion, dtype: float64

Train Set: 2000 samples | Test Set: 500 samples
Feature Matrix Dimension: 46 features


## 4. Model Selection & Training: Gradient Boosting vs Random Forest

In [4]:
print("🚀 Model 1: Training Gradient Boosting Classifier...")
if xgb is not None:
    gb_model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.08, max_depth=5, random_state=42, eval_metric="mlogloss")
else:
    gb_model = HistGradientBoostingClassifier(max_iter=100, random_state=42)

gb_model.fit(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

print("\n🌲 Model 2: Training Random Forest Classifier...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("✅ Both models trained successfully!")

🚀 Model 1: Training Gradient Boosting Classifier...

🌲 Model 2: Training Random Forest Classifier...
✅ Both models trained successfully!


## 5. Model Evaluation & Performance Comparison

In [5]:
all_classes = [0, 1, 2, 3]
all_names = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]

print("=== Gradient Boosting Performance Metrics ===")
print(classification_report(y_test, y_pred_gb, labels=all_classes, target_names=all_names, zero_division=0))
macro_f1_gb = f1_score(y_test, y_pred_gb, average="macro", zero_division=0)
print(f"Gradient Boosting Macro F1 Score: {macro_f1_gb:.4f}")

print("\n=== Random Forest Performance Metrics ===")
print(classification_report(y_test, y_pred_rf, labels=all_classes, target_names=all_names, zero_division=0))
macro_f1_rf = f1_score(y_test, y_pred_rf, average="macro", zero_division=0)
print(f"Random Forest Macro F1 Score: {macro_f1_rf:.4f}")

# Print Confusion Matrix text representation
cm = confusion_matrix(y_test, y_pred_gb, labels=all_classes)
print("\n=== Gradient Boosting Confusion Matrix ===")
print(pd.DataFrame(cm, index=[f"Actual_{l}" for l in all_names], columns=[f"Pred_{l}" for l in all_names]))

=== Gradient Boosting Performance Metrics ===
              precision    recall  f1-score   support

         LOW       0.00      0.00      0.00         0
      MEDIUM       0.80      0.44      0.57         9
        HIGH       0.87      0.91      0.89       247
    CRITICAL       0.76      0.79      0.78       195

   micro avg       0.82      0.85      0.83       451
   macro avg       0.61      0.54      0.56       451
weighted avg       0.82      0.85      0.83       451

Gradient Boosting Macro F1 Score: 0.6902

=== Random Forest Performance Metrics ===
              precision    recall  f1-score   support

         LOW       0.00      0.00      0.00         0
      MEDIUM       0.00      0.00      0.00         9
        HIGH       0.80      0.96      0.87       247
    CRITICAL       0.76      0.74      0.75       195

   micro avg       0.78      0.84      0.81       451
   macro avg       0.39      0.42      0.40       451
weighted avg       0.76      0.84      0.80       451



## 6. Feature Importance Analysis

In [6]:
print("💡 Top 10 Feature Importances (Random Forest):")
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances.head(10))

💡 Top 10 Feature Importances (Random Forest):
av_network              0.095792
av_physical             0.084385
pr_none                 0.077661
days_since_published    0.056913
ai_high                 0.055902
ac_high                 0.049286
ii_high                 0.044738
ac_low                  0.042736
ci_high                 0.040014
ui_required             0.037139
dtype: float64


## 7. Composite Risk Engine Scoring Test

In [7]:
print("🎯 Testing Composite Risk Prioritization Engine...")
scorer = RiskScorer()

sample_cve_id = "CVE-2021-44228"  # Log4Shell zero-day
result = scorer.score(
    cve_id=sample_cve_id,
    predicted_severity=3,  # CRITICAL
    model_confidence=0.98,
    in_cisa_kev=True,
    epss_score=0.99999,
    asset_criticality=0.9,
    exposure_score=1.0
)

print(f"\n--- Composite Risk Score for {sample_cve_id} ---")
print(f"Risk Score (0-100): {result.risk_score}")
print(f"Risk Tier:         {result.risk_tier}")
print("\nComponent Contributions:")
for component, contrib in result.component_contributions.items():
    print(f"  - {component:20s}: {contrib}%")

🎯 Testing Composite Risk Prioritization Engine...

--- Composite Risk Score for CVE-2021-44228 ---
Risk Score (0-100): 91.84
Risk Tier:         CRITICAL_RISK

Component Contributions:
  - cvss                : 23.4%
  - kev                 : 25.0%
  - epss                : 20.0%
  - asset_criticality   : 13.5%
  - exposure            : 7.0%
  - model_confidence    : 2.94%
